In [ ]:
import pathlib
import os 
import pandas as pd
import cv2
import numpy as np

train = "Dataset/Merged/Landmarks_3_10/train.txt"
val = "Dataset/Merged/Landmarks_3_10/val.txt"

train = pd.read_csv(train, header=None)
train = train.assign(split='train')
val = pd.read_csv(val, header=None)
val = val.assign(split='val')

train = pd.concat([train, val], ignore_index=True)
train = train.rename(columns={0: 'image_path'})

def get_dataset(image_path):
    if "HC" in image_path:
        return "HC18"
    elif len(pathlib.Path(image_path).stem) == 5:
        return "PHSFS"
    else:
        return "JNU-IFM"

train['dataset'] = train['image_path'].apply(get_dataset)

image_base = "Dataset/Merged/Landmarks_3_10/images"
mask_base = "Dataset/Merged/Landmarks_3_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset124_Fetal/imagesTr"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset124_Fetal/labelsTr"

os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in train.iterrows():
    print("\rProcessing %s/%s" % (i, len(train)), end="")
    image_path = os.path.join(image_base, row[1]['image_path'])
    mask_path = os.path.join(mask_base, row[1]['image_path'])
    
    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    # read both, path to square and resize to 512x512
    image = cv2.imread(image_path, 0)
    mask = cv2.imread(mask_path, 0)
    
    h, w = image.shape
    size = max(h, w)
    delta_w = size - w
    delta_h = size - h
    
    # pad with numpy
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)
    
    image = np.pad(image, ((top, bottom), (left, right)), mode='constant', constant_values=0)
    mask = np.pad(mask, ((top, bottom), (left, right)), mode='constant', constant_values=0)
     
    image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
    
    if image.shape != (512, 512) or mask.shape != (512, 512):
        raise ValueError("Image or mask not resized properly")
    
    cv2.imwrite(image_dest, image)
    cv2.imwrite(mask_dest, mask)

    i += 1

train = train.assign(image_reference=image_reference)
train.to_csv("fetal_train_with_reference.csv", index=False)

Processing 10262/10262

In [3]:
import pandas as pd
import os

# generate 4 splits for nnUNet, use the train - val split as is
# then they have 4 models, one uses each individual dataset as train/val
# final 4 split uses all datasets
# do not use scikit learn's train_test_split those are already defined
train = pd.read_csv("fetal_train_with_reference.csv")
val = train[train['split'] == 'val']
train = train[train['split'] == 'train']

folds = []
# split 0: HC18 train, val
dictionary = {'train': train[train['dataset'] == 'HC18']['image_reference'].tolist(),
              'val': val[val['dataset'] == 'HC18']['image_reference'].tolist()}
folds.append(dictionary)
# split 1: PHSFS train, val
dictionary = {'train': train[train['dataset'] == 'PHSFS']['image_reference'].tolist(),
              'val': val[val['dataset'] == 'PHSFS']['image_reference'].tolist()}
folds.append(dictionary)
# split 2: JNU-IFM train, val
dictionary = {'train': train[train['dataset'] == 'JNU-IFM']['image_reference'].tolist(),
              'val': val[val['dataset'] == 'JNU-IFM']['image_reference'].tolist()}
folds.append(dictionary)
# split 3: all train, val
dictionary = {'train': train['image_reference'].tolist(),
              'val': val['image_reference'].tolist()}
folds.append(dictionary)

# save to json
import json
os.makedirs('/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_preprocessed/Dataset124_Fetal', exist_ok=True)
with open('/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_preprocessed/Dataset124_Fetal/splits_final.json', 'w') as f:
    json.dump(folds, f)

In [ ]:
import pathlib
import os 
import pandas as pd
import cv2

train = "Dataset/Merged/Landmarks_3_10/test.txt"

train = pd.read_csv(train, header=None)
train = train.assign(split='test')
train = train.rename(columns={0: 'image_path'})

def get_dataset(image_path):
    if "HC" in image_path:
        return "HC18"
    elif len(pathlib.Path(image_path).stem) == 5:
        return "PSFHS"
    else:
        return "JNU-IFM"

train['dataset'] = train['image_path'].apply(get_dataset)

image_base = "Dataset/Merged/Landmarks_3_10/images"
mask_base = "Dataset/Merged/Landmarks_3_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset124_Fetal/imagesTs"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset124_Fetal/labelsTs"

os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in train.iterrows():
    print("\rProcessing %s/%s" % (i, len(train)), end="")
    image_path = os.path.join(image_base, row[1]['image_path'])
    mask_path = os.path.join(mask_base, row[1]['image_path'])
    
    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    # read both, path to square and resize to 512x512
    image = cv2.imread(image_path, 0)
    mask = cv2.imread(mask_path, 0)

    h, w = image.shape[:2]
    size = max(h, w)
    delta_w = size - w
    delta_h = size - h
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)
    image = np.pad(image, ((top, bottom), (left, right)), mode='constant', constant_values=0)
    mask = np.pad(mask, ((top, bottom), (left, right)), mode='constant', constant_values=0)
    image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
    
    if image.shape != (512, 512) or mask.shape != (512, 512):
        raise ValueError("Image or mask not resized properly")
    
    cv2.imwrite(image_dest, image)
    cv2.imwrite(mask_dest, mask)

    i += 1

train = train.assign(image_reference=image_reference)
train.to_csv("fetal_test_with_reference.csv", index=False)

Processing 1041/1041

In [4]:
from medpy.metric.binary import dc, hd, __surface_distances
import numpy as np
import cv2 
import pandas as pd
import os

def calculate_metrics(pred, gt):    
    dc_val = dc(pred, gt)
    if np.sum(pred) == 0:
        hd_val = np.nan
        assd_val = np.nan
        return dc_val, hd_val, assd_val
    hd_val = hd(pred, gt, voxelspacing=(1, 1))
    d1 = __surface_distances(pred, gt, voxelspacing=(1, 1))
    d2 = __surface_distances(gt, pred, voxelspacing=(1, 1))
    assd_val = np.mean(np.concatenate((d1, d2)))
    return dc_val, hd_val, assd_val

test = pd.read_csv("fetal_test_with_reference.csv")

organs = [1,2]
organ_names = ['Pubic Symphysis', 'Fetal Head']

nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset124_Fetal/labelsTs"
nnUNet_pred_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_results/Dataset124_Fetal/nnUNetTrainer__nnUNetPlans__2d"

test['label_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_mask_base, x + ".png"))
results = []

for f in range(4):
    test['pred_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_pred_base, "fold_%s/test"%f, x + ".png"))
    
    if f == 0:
        fold = "HC18"
    elif f == 1:
        fold = "PSFHS"
    elif f == 2:
        fold = "JNU-IFM"
    elif f == 3:
        fold = "All"
    
    for row in test.iterrows():
        print(row[1])

        GT = cv2.imread(row[1]['label_path'], cv2.IMREAD_UNCHANGED)
        mask = cv2.imread(row[1]['pred_path'], cv2.IMREAD_UNCHANGED)

        for organ_num, organ_name in zip(organs, organ_names):
            pred = (mask == int(organ_num)).astype(np.uint8)
            gt = (GT == int(organ_num)).astype(np.uint8)
            
            if np.sum(gt) == 0:
                continue

            dc_val, hd_val, assd_val = calculate_metrics(pred, gt)
            results.append({
                "image": row[1]['image_path'],
                "organ": organ_name,
                "dc": dc_val,
                "hd": hd_val,
                "assd": assd_val,
                "fold": fold
            })
        print(results[-1])

results = pd.DataFrame(results)
results.to_csv("fetal_head_nnunet_results.csv", index=False)

image_path                                                159_HC.png
split                                                           test
dataset                                                         HC18
image_reference                                              image_1
label_path         /media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnU...
pred_path          /media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnU...
Name: 0, dtype: object
{'image': '159_HC.png', 'organ': 'Fetal Head', 'dc': 0.9775669652535097, 'hd': np.float64(9.219544457292887), 'assd': np.float64(2.8045427173042277), 'fold': 'HC18'}
image_path                                                461_HC.png
split                                                           test
dataset                                                         HC18
image_reference                                              image_2
label_path         /media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnU...
pred_path          /media/ngaggion/SSDNVME/APOLO/nnU

(np.float64(0.8349907137146545),
 np.float64(41.175363841033324),
 np.float64(11.283386542407863))